# Week 2, Lab 3 — Handoffs\n\nTriage routes to a math specialist or a facts specialist.


In [ ]:
WEEK = 'Week 2'
LAB = 'Lab 3 — handoffs'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn openai openai-agents
else:
    %pip install -q ollama openai openai-agents


In [ ]:
cfg = openai_client_kwargs()
print(cfg)

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, handoff

client = AsyncOpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])
model = OpenAIChatCompletionsModel(model=cfg["model"], openai_client=client)


In [ ]:
from agents import function_tool, handoff

@function_tool
def calculator_tool(expression: str) -> str:
    """Evaluate arithmetic."""
    return calculator(expression)

@function_tool
def lookup_fact_tool(topic: str) -> str:
    """Local course facts."""
    return lookup_fact(topic)

math_agent = Agent(
    name="MathSpecialist",
    instructions="You only solve math. Always use calculator_tool.",
    model=model,
    tools=[calculator_tool],
)
facts_agent = Agent(
    name="FactsSpecialist",
    instructions="You only answer with lookup_fact_tool.",
    model=model,
    tools=[lookup_fact_tool],
)
triage = Agent(
    name="Triage",
    instructions="If the user asks math, hand off to MathSpecialist. If they ask about AI/agents topics, hand off to FactsSpecialist. Otherwise answer briefly yourself.",
    model=model,
    handoffs=[handoff(math_agent), handoff(facts_agent)],
)

for q in ["What is 19*21?", "What is Ollama?", "Say hello."]:
    result = await Runner.run(triage, q)
    print("Q:", q)
    print("A:", result.final_output)
    print("---")


## Exercise\n\nAdd a DateAgent specialist.\n\n**Next:** guardrails.
